# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alao64/FlyRank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

### Growth/Recovery Prediction


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/Alao64/FlyRank/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("trend_direction class sizes:")
print(df['trend_direction'].value_counts())

trend_pct_dist = (df['trend_direction'].value_counts(normalize=True) * 100).round(1)
print()
print("trend_direction distribution (%):")
print(trend_pct_dist)

trend_direction class sizes:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_direction distribution (%):
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8
Name: proportion, dtype: float64


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The question: decision, action, cost of a wrong call

**Search question:** Can we predict which pages are likely to decline, recover, or gain momentum —
using a prior feature window to predict a future target window

**Unit of analysis:** a single content item (a page), evaluated at a point in time, with a defined
prior feature window and a later target window.

**The decision this improves:** which pages a content/SEO reviewer should look at first each week
— right now that's decided reactively (someone notices a page's traffic already dropped), instead
of proactively flagging pages whose *trajectory* suggests they're about to decline, or that a past
refresh is starting to pay off.

**Who acts, and what they do:** a content strategist or SEO reviewer works down a ranked queue of
flagged pages and decides whether to refresh, consolidate, or leave each one alone.

**Cost of a wrong call:**
- *False negative* (a real decline missed): the page keeps losing impressions/clicks unnoticed until someone catches it by chance
- *False positive* (a page flagged as declining/gaining that wasn't): wasted reviewer time on a page that didn't need attention

**Why data/ML helps at all:** a simple rule (e.g. "clicks fell") can catch obvious cases, but it
can't separate a real decline from consolidation, seasonality, or noise, and it can't combine many
weaker, tangled signals (position drift, engagement rate, freshness, content type, age) into one
score the way a model can. ML only earns its place here if it beats that kind of simple rule on a
held-out, client-grouped test set.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Sanity check the "why ML helps" claim: how far does a one-line rule already get us?
comparable = df[(df['clicks_prev_30d'] > 0)]
rule_flag = comparable['clicks_last_30d'] < comparable['clicks_prev_30d']
actual_down = comparable['trend_direction'] == 'down'

agreement = (rule_flag == actual_down).mean() * 100
print(f"Rows where both 30d click windows exist: {len(comparable)}")
print(f"Simple 'clicks fell' rule agrees with labeled trend_direction=='down': {agreement:.1f}% of the time")
print("(Rough sanity check, not a real baseline — shows the rule alone isn't a perfect proxy.)")

Rows where both 30d click windows exist: 11759
Simple 'clicks fell' rule agrees with labeled trend_direction=='down': 61.3% of the time
(Rough sanity check, not a real baseline — shows the rule alone isn't a perfect proxy.)


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

## 3. Quick look at the data (2-3 real numbers)

Numbers pulled from `data/raw/content_refresh_anonymized.csv` (30,000 rows, 32 clients,
trailing-90-day metrics per content item):

1. **54.2% of rows are labeled `down`** in `trend_direction`, versus 19.9% `stable`, 14.6% `up`,
   7.5% `new`, and 3.8% `flat`. Decline is the majority class here — plenty of positive examples,
   but it also means a lazy "always predict down" baseline would already score ~54% accuracy, so
   I'll need a real baseline (not just accuracy) to prove a model adds value.
2. **84.1% of rows (25,217 of 30,000) have nonzero impressions in *both* the prior and last 30-day
   windows** — a genuine prior-window-to-later-window comparison exists for the large majority of
   the data, which is the structural requirement for this lane's "prior window → future window"
   label shape.
3. **Client volume is extremely uneven**: 32 clients, ranging from 3 rows to 7,008 rows per
   client (median 567). Any train/test split has to be **grouped by `client_id`**, or the model
   will leak client-specific patterns between train and test.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
both_windows = ((df['impressions_prev_30d'] > 0) & (df['impressions_last_30d'] > 0)).mean() * 100
print(f"Rows with a usable prior AND last 30-day window: {both_windows:.1f}%")
print()

per_client = df.groupby('client_id').size()
print(f"Clients: {df['client_id'].nunique()}, rows per client: min={per_client.min()}, "
      f"median={per_client.median():.0f}, max={per_client.max()}")

Rows with a usable prior AND last 30-day window: 84.1%

Clients: 32, rows per client: min=3, median=567, max=7008


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful words: what I can and can't claim

**What I can claim, if the work holds up:**
- *Observed*: which content items had a measured decline, stable, or growth outcome in a defined
  future window, based on the data as recorded.
- *Directional / decision-support*: a ranked list of "review this first" candidates, with reason
  codes drawn from features that existed **before** the outcome window — a prioritization aid for
  a human reviewer, not a certainty.
- *Comparative*: whether a model beats a simple rule-based baseline on a held-out, client-grouped
  test set, using a metric named in advance (precision@K, recall on the decline class).

**What I will not claim:**
- I will **not** claim a refresh *caused* a recovery — that needs an experiment or causal design,
  which this observational data can't give me on its own.
- I will **not** claim to predict Google's algorithm, ranking factors, or AI citations — the data
  only contains this product's own observable, anonymized signals.
- I will **not** treat `trend_direction` / `trend_pct` as clean ground truth without first checking
  whether a "decline" is actually consolidation, seasonality, or noise.
- I will **not** use `trend_direction` or `trend_pct` as model *features* — they're derived from
  the future window I'm trying to predict, which would be leakage, not discovery.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("No new numbers needed here  see sections above for the supporting evidence.")

No new numbers needed here  see sections above for the supporting evidence.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.